# Task 8 — SSA trên doanh số gốc

Chạy từ trên xuống bằng CPU trên Colab hoặc máy cá nhân. Dùng hai Parquet đã đối soát ở Task 6. Notebook kiểm tra SHA-256 trước khi đọc và chỉ ghi vào thư mục kết quả.

Công thức dự báo và tổng hợp theo ngày giữ theo [SSA của tác giả, commit 8af694a](https://github.com/Dingdong-Inc/frn-50k-baseline/blob/8af694a09a5a427d1d50648d348c022565a781f2/demand_forecasting/SSA/ssa_forecasting.py). Kết quả chính gồm ba nhóm Overall, Low Sale và High Sale. Số tham chiếu lấy từ [Bảng 3 của bài báo](https://arxiv.org/html/2505.16319v5#S5.SS3).

In [1]:
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
import json
import platform
import os

import numpy as np
import pandas as pd
import pyarrow as pa

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount('/content/drive')
    project_dir = Path('/content/drive/MyDrive/PTDLKD')  # Sửa theo tên thư mục Drive thực tế.
    data_dir = project_dir / 'data'
    output_dir = project_dir / 'task08_outputs'
else:
    cwd = Path.cwd().resolve()
    project_dir = next((p for p in (cwd, *cwd.parents)
                        if (p / 'Task08_SSA_Raw_Sales').is_dir()), cwd)
    data_dir = project_dir
    output_dir = project_dir / 'Task08_SSA_Raw_Sales/results'

# Có thể chỉnh hai đường dẫn này nếu dữ liệu được đặt ở thư mục khác.
data_dir = Path(os.environ.get('TASK08_DATA_DIR', data_dir))
output_dir = Path(os.environ.get('TASK08_OUTPUT_DIR', output_dir))

## Xác nhận dữ liệu đầu vào

SHA-256 tham chiếu lấy từ biên bản Task 6. Nếu không khớp, dùng đúng bản đã xác nhận hoặc đối soát lại ở Task 6 trước khi thay tham chiếu. Đọc tám cột cần thiết; không tải toàn bộ các mảng giờ vào RAM.

In [2]:
EXPECTED_SHA256 = {'train': 'b7e01c711bc4fa0598c4a8cf4e8b97769275b1afa1792d98fd8428c76838bc96', 'eval': '61cfd3cb60d767d38ba9e640be64e982a15b2918ca0ba14fe0283c5a2eac6872'}

COLS = ['store_id', 'product_id', 'dt', 'sale_amount', 'stock_hour6_22_cnt',
        'discount', 'holiday_flag', 'precpt']

def sha256_file(path):
    digest = sha256()
    with path.open('rb') as file:
        for block in iter(lambda: file.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

inputs = {}
for split in ('train', 'eval'):
    path = data_dir / f'{split}.parquet'
    if not path.is_file():
        raise FileNotFoundError(f'Thiếu {path}. Chỉnh data_dir và chọn hai file từ Task 6.')
    observed = sha256_file(path)
    if observed != EXPECTED_SHA256[split]:
        raise ValueError(f'SHA-256 của {split}.parquet khác Task 6: {observed}. '
                         'Dùng đúng bản đã xác nhận hoặc chạy lại Task 6.')
    inputs[split] = {'filename': path.name, 'sha256': observed, 'bytes': path.stat().st_size}
    print(f'{split}.parquet: SHA-256 khớp Task 6.')

train_df = pd.read_parquet(data_dir / 'train.parquet', columns=COLS, engine='pyarrow')
eval_df = pd.read_parquet(data_dir / 'eval.parquet', columns=COLS, engine='pyarrow')
if (len(train_df), len(eval_df)) != (4_500_000, 350_000):
    raise ValueError('Số dòng train/eval không đúng Task 6.')
inputs['train']['rows'] = len(train_df)
inputs['eval']['rows'] = len(eval_df)
eligible_eval = int((eval_df['stock_hour6_22_cnt'] == 0).sum())
print(f'Train: {len(train_df):,}; Eval: {len(eval_df):,}; Eval đủ điều kiện: {eligible_eval:,}.')

train.parquet: SHA-256 khớp Task 6.
eval.parquet: SHA-256 khớp Task 6.
Train: 4,500,000; Eval: 350,000; Eval đủ điều kiện: 206,562.


## Dự báo và đánh giá

Giữ 90 ngày train làm lịch sử, dự báo 7 ngày eval. Tính psd từ doanh số gốc của train theo từng cặp. Lọc eval bằng stock_hour6_22_cnt == 0, tính WAPE/WPE/MAE theo ngày rồi lấy trung bình. Dự báo chỉ dùng doanh số lịch sử; doanh số eval dùng để đối chiếu.

Các số (%) của bài báo được lưu riêng, chỉ dùng để tính chênh lệch điểm phần trăm sau khi có kết quả tự chạy.

In [3]:
all_data = pd.concat([train_df, eval_df], axis=0, ignore_index=True)
del train_df, eval_df  # Giải phóng RAM ngay lập tức

all_data = all_data.sort_values(by=['store_id', 'product_id', 'dt']).reset_index(drop=True)

# 2. Tạo đặc trưng lịch và thời tiết giống hệt ssa_forecasting.py
dt_series = pd.to_datetime(all_data['dt'])
all_data['time_idx'] = (dt_series - dt_series.min()).dt.days
all_data['dow'] = dt_series.dt.dayofweek
all_data.loc[all_data['dt'].isin(['2024-05-02', '2024-05-03', '2024-05-04']), 'holiday_flag'] = 0
all_data.loc[all_data['holiday_flag'] == 1, 'dow'] = 5
all_data.loc[(all_data['holiday_flag'] == 0) & (all_data['dow'].isin([5, 6])), 'dow'] = 0
all_data['is_precpt'] = (all_data['precpt'] > 3.5).astype(int)

# 3. Tính trọng số SSA
x_cont = all_data[['time_idx', 'discount', 'sale_amount']].values.reshape(-1, 97, 3)
x_cat = all_data[['dow', 'holiday_flag', 'is_precpt']].values.reshape(-1, 97, 3)

max_encoder_length = 90
holiday_label = 5

date_distance = 18 - x_cont[:, :max_encoder_length, 0][:, None, :] / 5
discount_distance = 1 - np.abs(x_cont[:, max_encoder_length:, 1][..., None] - x_cont[:, :max_encoder_length, 1][:, None, :])

decoder_dow = x_cat[:, max_encoder_length:, 0][..., None]
encoder_dow = x_cat[:, :max_encoder_length, 0][:, None, :]
dow_distance = decoder_dow - encoder_dow

decoder_precpt = x_cat[:, max_encoder_length:, 2][..., None]
encoder_precpt = x_cat[:, :max_encoder_length, 2][:, None, :]
precpt_distance = decoder_precpt - encoder_precpt

date_distance = np.where((decoder_dow == holiday_label).astype(int) - (encoder_dow == holiday_label).astype(int) == 0, date_distance - 3, date_distance)
date_distance = np.where((decoder_dow != holiday_label) & (encoder_dow == holiday_label), date_distance + 3, date_distance)
date_distance = np.where(dow_distance == 0, date_distance - 3, date_distance)
date_distance = np.where((precpt_distance == 0) & (decoder_precpt == 1), date_distance - 3, date_distance)
date_distance = np.where(precpt_distance != 0, date_distance + 3, date_distance)
date_distance = np.clip(date_distance, a_min=1, a_max=18)

del dow_distance, precpt_distance, decoder_precpt, encoder_precpt
weight = (18 - date_distance) * (discount_distance ** 2)
del date_distance, discount_distance
weight = np.exp(weight) / np.exp(weight).sum(axis=-1, keepdims=True)

# 4. Chia nhóm High Sale / Low Sale dựa trên psd
mu = all_data[all_data['dt'] <= '2024-06-25'].groupby(['store_id', 'product_id'], as_index=False)['sale_amount'].mean()
mu.rename(columns={'sale_amount': 'psd'}, inplace=True)

pdf = all_data[all_data['dt'] >= '2024-06-26'].copy()
pdf = pdf.merge(mu, on=['store_id', 'product_id'], how='left')

target = x_cont[:, :max_encoder_length, 2:3]
index = encoder_dow.squeeze()[..., None] == np.arange(6)
mean_sale = np.nanmean(np.where(index, target, np.nan), axis=1) + 0.001
season_ratio = np.nanmedian(mean_sale[..., None] / mean_sale[:, None, ...], axis=0)
ratio = season_ratio[decoder_dow, encoder_dow]
pred = (weight * ratio) @ target
pdf['ssa_pred_sale_amount'] = pred.reshape(-1)
del weight, ratio, x_cont, x_cat, all_data, pred


def evaluate_group(df_eval_pred, condition):
    wape_list, mae_list, wpe_list = [], [], []
    sub = df_eval_pred.query('stock_hour6_22_cnt == 0').query(condition)
    if sub['dt'].nunique() != 7:
        raise ValueError(f'{condition}: không đủ 7 ngày đánh giá.')
    for dt, subdf in sub.groupby('dt'):
        if subdf['sale_amount'].sum() <= 0 or not np.isfinite(subdf['ssa_pred_sale_amount']).all():
            raise ValueError(f'{condition}, ngày {dt}: dự báo không hữu hạn hoặc tổng doanh số không dương.')
        abs_err = (subdf['sale_amount'] - subdf['ssa_pred_sale_amount']).abs()
        wape_list.append(abs_err.sum() / subdf['sale_amount'].sum())
        mae_list.append(abs_err.mean())
        wpe_list.append((subdf['ssa_pred_sale_amount'] - subdf['sale_amount']).sum() / subdf['sale_amount'].sum())
    return round(np.mean(wape_list), 4), round(np.mean(wpe_list), 4), round(np.mean(mae_list), 4)

paper_ref = {
    "Overall":   {"cond": "psd>=0", "wape": 31.97, "wpe": -10.50},
    "Low Sale":  {"cond": "psd<1",  "wape": 38.33, "wpe": -8.03},
    "High Sale": {"cond": "psd>=1", "wape": 27.40, "wpe": -12.26}
}

log_lines = ["group\tdemand\twape\twpe\tmae"]
csv_rows = []

for group_name, info in paper_ref.items():
    wape, wpe, mae = evaluate_group(pdf, info["cond"])
    log_lines.append(f"{info['cond']}\tssa_pred_sale_amount\t{wape}\t{wpe}\t{mae}")

    wape_pct = round(wape * 100, 2)
    wpe_pct = round(wpe * 100, 2)
    csv_rows.append({
        "config_id": "SSA_RawSales",
        "group": group_name,
        "wape_pct": wape_pct,
        "wpe_pct": wpe_pct,
        "mae": mae,
        "paper_wape_pct": info["wape"],
        "paper_wpe_pct": info["wpe"],
        "delta_wape_pp": round(wape_pct - info["wape"], 2),
        "delta_wpe_pp": round(wpe_pct - info["wpe"], 2),
        "status": "completed"
    })

## Lưu bản bàn giao

CSV kết quả phải có đủ ba dòng; log lưu tỷ lệ WAPE/WPE trước khi đổi sang phần trăm. JSON ghi phiên bản thư viện và dấu vân tay của đầu vào. Trên Colab, mặc định file kết quả nằm trên Google Drive.

In [4]:
if len(csv_rows) != 3 or not np.isfinite(
        [[row[k] for k in ('wape_pct', 'wpe_pct', 'mae')] for row in csv_rows]).all():
    raise ValueError('Kết quả bàn giao phải có đúng ba dòng với chỉ số hữu hạn.')

(output_dir / 'full').mkdir(parents=True, exist_ok=True)
log_content = '\n'.join(log_lines) + '\n'
(output_dir / 'full/ssa_raw_sales.log').write_text(log_content, encoding='utf-8')
df_metrics = pd.DataFrame(csv_rows)
df_metrics.to_csv(output_dir / 'ssa_raw_sales_metrics.csv', index=False, encoding='utf-8', lineterminator='\n')
metadata = {
    'config_id': 'SSA_RawSales',
    'run_utc': datetime.now(timezone.utc).isoformat(),
    'runtime': 'Colab CPU' if drive is not None else f'{platform.system()} CPU',
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'pyarrow': pa.__version__,
    'ssa_reference_commit': '8af694a09a5a427d1d50648d348c022565a781f2',
    'inputs': inputs,
    'eligible_eval_rows': eligible_eval,
    'metrics_rows': len(df_metrics),
    'metric_units': {'wape_pct': 'percent', 'wpe_pct': 'percent', 'mae': 'normalized sales'},
}
(output_dir / 'run_metadata.json').write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + '\n',
                                               encoding='utf-8')
print(log_content, end='')
print(df_metrics.to_string(index=False))
print('Đã lưu CSV, log và metadata vào thư mục kết quả.')

group	demand	wape	wpe	mae
psd>=0	ssa_pred_sale_amount	0.3197	-0.105	0.3982
psd<1	ssa_pred_sale_amount	0.3833	-0.0803	0.2691
psd>=1	ssa_pred_sale_amount	0.274	-0.1226	0.7761
   config_id     group  wape_pct  wpe_pct    mae  paper_wape_pct  paper_wpe_pct  delta_wape_pp  delta_wpe_pp    status
SSA_RawSales   Overall     31.97   -10.50 0.3982           31.97         -10.50            0.0           0.0 completed
SSA_RawSales  Low Sale     38.33    -8.03 0.2691           38.33          -8.03            0.0           0.0 completed
SSA_RawSales High Sale     27.40   -12.26 0.7761           27.40         -12.26            0.0           0.0 completed
Đã lưu CSV, log và metadata vào thư mục kết quả.
